In [1]:
!pip install requests beautifulsoup4 nltk pymupdf gradio pandas matplotlib scikit-learn numpy firebase datasets google-genai huggingface_hub tensorflow pillow -q


In [2]:
import requests
from bs4 import BeautifulSoup
import re, json, base64, io, tempfile
import uuid as _uuid
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gradio as gr
from datetime import datetime, timedelta
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem import WordNetLemmatizer
import nltk
import fitz
from firebase import firebase
from PIL import Image
from huggingface_hub import hf_hub_download
import tensorflow as tf
from google import genai
nltk.download('wordnet', quiet=True)
print("All imports loaded successfully")


All imports loaded successfully


# Firebase Connection

In [3]:
# Connect to Firebase and define the database paths used in the project.
# These paths store the search index, sensor readings, uploaded images,
# treatment cases, and gamification data.

FIREBASE_URL = "https://bluberrybee1-default-rtdb.firebaseio.com/"
FBconn = firebase.FirebaseApplication(FIREBASE_URL, None)
INDEX_PATH        = "/blueberry_index"
SENSOR_CACHE_PATH = "/sensor_cache"
IMAGES_PATH       = "/uploaded_images"
CASES_PATH        = "/treatment_cases"
GAMIFICATION_PATH = "/gamification"
print("Connected to Firebase Realtime Database")


Connected to Firebase Realtime Database


In [4]:
# Create the Gemini client that will be used throughout the project
# to generate responses and recommendations.

client = genai.Client(api_key="AQ.Ab8RN6Ly5s-ZiworgAB3EfRWKbpImi5X4jyDZgUpC5tIXco5fg")
print("Gemini client ready")

Gemini client ready


## Big Data: MapReduce Pattern

In this part we use the MapReduce idea from the course to build an inverted index.
This index helps us quickly find which papers contain each important word.

In [5]:
# Build an inverted index using the MapReduce approach.
# The text is cleaned, grouped by words, and then stored
# so we can search the blueberry papers more efficiently.

# Common words that we do not want to include in the index
STOP_WORDS = {
    'a','an','the','and','or','in','on','at','to','for','of','with',
    'is','are','was','were','be','been','being','have','has','had',
    'do','does','did','will','would','could','should','may','might',
    'this','that','these','those','by','from','as','it','its','not',
    'but','we','our','their','they','he','she','his','her'
}
lemmatizer = WordNetLemmatizer()


def map_phase(doc_id, soup):
    # Creates pairs of (word, document id) from one paper
    if soup is None:
        return []
    pairs = []
    for w in re.findall(r'\w+', soup.get_text()):
        w = w.lower()
        if w not in STOP_WORDS and len(w) > 2:
            pairs.append((lemmatizer.lemmatize(w), doc_id))
    return pairs


def shuffle_phase(pairs):
    # Groups all document IDs under the same word
    g = {}
    for w, d in pairs:
        g.setdefault(w, []).append(d)
    return g


def reduce_phase(grouped):
    # Removes duplicate document IDs for each word
    return {w: sorted(set(ids)) for w, ids in grouped.items()}


def build_index_mapreduce(pages):
    # Runs the full process: map, shuffle, and reduce
    print("[MapReduce] MAP phase...")
    all_pairs = []
    for doc_id, soup in pages.items():
        pairs = map_phase(doc_id, soup)
        all_pairs.extend(pairs)
        print(f"  doc {doc_id}: {len(pairs)} pairs")
    print(f"[MapReduce] SHUFFLE ({len(all_pairs)} pairs)...")
    grouped = shuffle_phase(all_pairs)
    print(f"[MapReduce] REDUCE ({len(grouped)} terms)...")
    index = reduce_phase(grouped)
    print(f"[MapReduce] Done -> {len(index)} terms in index")
    return index


def build_paper_word_counts(pages):
    # Counts how many times each word appears in each paper
    out = {}
    for doc_id, soup in pages.items():
        idx = {}
        if soup:
            for w in re.findall(r'\w+', soup.get_text()):
                w = w.lower()
                if w not in STOP_WORDS:
                    lw = lemmatizer.lemmatize(w)
                    idx[lw] = idx.get(lw, 0) + 1
        out[doc_id] = idx
    return out


print("Big Data MapReduce functions ready")


Big Data MapReduce functions ready


## Microservice Architecture

In this part we separate the system into three microservices.
Each one is responsible for a different task: sensor data, AI responses, or database operations.

In [6]:
# Each microservice is responsible for one part of the system.
# This keeps the code more organized and easier to maintain.

class SensorMicroservice:
    # Handles requests to the sensor server
    BASE = "https://server-cloud-v645.onrender.com"

    @staticmethod
    def get_history(feed, limit, retries=2, timeout=60):
        # Gets sensor history from the cloud server
        last_err = None
        for attempt in range(retries + 1):
            try:
                r = requests.get(
                    f"{SensorMicroservice.BASE}/history",
                    params={"feed": feed, "limit": limit},
                    timeout=timeout
                )
                r.raise_for_status()
                data = r.json()
                if "data" not in data:
                    raise ValueError(f"Bad response: {data}")
                return data["data"]
            except Exception as e:
                last_err = e
                if attempt < retries:
                    print(f"  [sensor] attempt {attempt+1} failed, retrying... ({e})")
        raise last_err


class GeminiMicroservice:
    # Sends prompts to the Gemini model
    MODEL = "gemini-2.5-flash"

    def __init__(self, c):
        self._c = c

    def generate(self, prompt):
        # Sends a prompt to Gemini and returns the response
        return self._c.models.generate_content(
            model=self.MODEL, contents=prompt
        ).text

    def generate_json(self, prompt):
        # Returns the model response as a JSON object
        t = self.generate(prompt).replace("```json", "").replace("```", "").strip()
        return json.loads(t)


class FirebaseMicroservice:
    # Handles reading and writing data in Firebase
    def __init__(self, conn):
        self._conn = conn

    def put(self, path, key, data):
        return self._conn.put(path, key, data)

    def post(self, path, data):
        return self._conn.post(path, data)

    def get(self, path, key=None):
        return self._conn.get(path, key)

    def delete(self, path, key=None):
        return self._conn.delete(path, key)

# Create one instance of each microservice
sensor_svc = SensorMicroservice()
gemini_svc = GeminiMicroservice(client)
fb_svc     = FirebaseMicroservice(FBconn)
print("Microservices initialized")


Microservices initialized


## Search Engine
### Fetch Academic Papers

In this part we define the academic papers that will be used in the search engine.
The code downloads each paper and converts its content into text, so we can process it later.

In [7]:
# Links to the papers we use as the data source for the project
PAPERS = {
    1: 'https://doi.org/10.3389/fpls.2024.1428769',
    2: 'https://doi.org/10.24266/0738-2898-33.1.33',
    3: 'http://www.globalsciencebooks.info/Online/GSBOnline/images/0706/EJPSB_1(1)/EJPSB_1(1)44-56o.pdf',
    4: 'https://journals.plos.org/plosone/article/file?id=10.1371/journal.pone.0283137&type=printable',
    5: 'https://doi.org/10.3389/fpls.2015.00782'
}

def fetch_page(url):
    # Downloads a web page or PDF and returns its text content
    try:
        r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'},
                         timeout=10, allow_redirects=True)
        if r.status_code == 200:
            ct = r.headers.get('Content-Type', '')
            if 'application/pdf' in ct or url.endswith('.pdf'):
                doc = fitz.open(stream=r.content, filetype='pdf')
                return BeautifulSoup(''.join(p.get_text() for p in doc), 'html.parser')
            return BeautifulSoup(r.text, 'html.parser')
    except Exception as e:
        print(f'  Error: {e}')
    return None

# Fetch all papers and save the extracted content in a dictionary
print('Fetching papers...')
pages = {}
for doc_id, url in PAPERS.items():
    print(f'Paper {doc_id}: {url}')
    pages[doc_id] = fetch_page(url)
    print(f"  {'OK' if pages[doc_id] else 'FAILED'}")


Fetching papers...
Paper 1: https://doi.org/10.3389/fpls.2024.1428769
  OK
Paper 2: https://doi.org/10.24266/0738-2898-33.1.33
  OK
Paper 3: http://www.globalsciencebooks.info/Online/GSBOnline/images/0706/EJPSB_1(1)/EJPSB_1(1)44-56o.pdf
  OK
Paper 4: https://journals.plos.org/plosone/article/file?id=10.1371/journal.pone.0283137&type=printable
  OK
Paper 5: https://doi.org/10.3389/fpls.2015.00782
  OK


### Build Index via MapReduce

In this step we build the word counts for each paper and create the full inverted index.
The index will be used later to retrieve relevant papers for a user query.

In [8]:
# Build the paper word counts and the full inverted index
paper_indices = build_paper_word_counts(pages)
for doc_id, idx in paper_indices.items():
    print(f'Paper {doc_id}: {len(idx)} unique terms')
full_inverted_index = build_index_mapreduce(pages)
print(f"\nFull inverted index: {len(full_inverted_index)} terms")


Paper 1: 3945 unique terms
Paper 2: 3132 unique terms
Paper 3: 3349 unique terms
Paper 4: 1517 unique terms
Paper 5: 3267 unique terms
[MapReduce] MAP phase...
  doc 1: 11672 pairs
  doc 2: 9476 pairs
  doc 3: 9381 pairs
  doc 4: 3603 pairs
  doc 5: 8572 pairs
[MapReduce] SHUFFLE (42704 pairs)...
[MapReduce] REDUCE (9399 terms)...
[MapReduce] Done -> 9399 terms in index

Full inverted index: 9399 terms


### Firebase Index Save/Load

In this part we create the main search index and store it in Firebase.
If the index already exists, we load it instead of rebuilding it.

In [9]:
# Terms used to build the main search index
TERMS = [
    'blueberry','vaccinium','anthocyanin','phenolic','cultivar',
    'ripening','maturity','harvest','quality','disease',
    'botrytis','anthracnose','colletotrichum','rot','resistance',
    'breeding','irrigation','fertilizer','drought','soil'
]

# Saves the index to Firebase
def save_index_to_firebase(index):
    FBconn.delete(INDEX_PATH, None)
    for term, data in index.items():
        FBconn.put(INDEX_PATH, term, {"term": data["term"], "DocIDs": data["DocIDs"]})
    print("Index saved to Firebase.")

# Loads the index from Firebase if it exists
def load_index_from_firebase():
    idx = FBconn.get(INDEX_PATH, None)
    if idx:
        print("Index loaded from Firebase.")
        return idx
    return None

# Creates the main index from the selected terms
def build_main_index(terms, paper_indices):
    main = {}
    for term in terms:
        lemma = lemmatizer.lemmatize(term.lower())
        doc_ids = [did for did, idx in paper_indices.items() if lemma in idx]
        main[term] = {'term': term, 'DocIDs': doc_ids}
    return main

# Load the index if it already exists, otherwise build and save it
main_index = load_index_from_firebase()
if main_index is None:
    print('Building index from papers...')
    main_index = build_main_index(TERMS, paper_indices)
    save_index_to_firebase(main_index)
print(f"Index ready: {len(main_index)} terms")


Index loaded from Firebase.
Index ready: 20 terms


### Search Function

This function searches the papers for the words in the user's query.
The matching papers are ranked based on the number of matching words.

In [10]:
# Searches the papers and ranks the matching results
def search(query, paper_indices):
    lem = WordNetLemmatizer()
    q_words = [lem.lemmatize(w) for w in re.findall(r'\w+', query.lower())]
    scores = {}
    for did, idx in paper_indices.items():
        found = {w: idx[w] for w in q_words if w in idx}
        if found:
            scores[did] = {'results': found, 'rank': 1 - 1 / (sum(found.values()) + 1)}
    return sorted(scores.items(), key=lambda x: x[1]['rank'], reverse=True)

print("Search function ready")


Search function ready


## RAG System

In this part we build the RAG system for the blueberry papers.
The system downloads the papers, splits them into smaller chunks, searches for the most relevant chunks, and then uses Gemini to generate an answer based on them.

In [11]:
# Install the libraries needed for PDF reading and Gemini
!pip install -q pymupdf requests google-genai

In [12]:
# Downloads a PDF file from a URL
def download_pdf(url, filename):
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    with open(filename, "wb") as f:
        f.write(r.content)
    return filename

# Extracts text from a PDF file
def extract_text_from_pdf(path):
    doc = fitz.open(path)
    return "".join(page.get_text() for page in doc)

# Splits long text into smaller chunks with some overlap
def split_text(text, chunk_size=900, overlap=150):
    text = re.sub(r"\s+", " ", text)
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks

# Removes chunks that do not contain enough real text
def is_garbage_chunk(chunk):
    non_alpha = sum(1 for c in chunk if c.isalpha())
    total     = max(len(chunk), 1)
    return (non_alpha / total) < 0.20

# A simple local vector store for saving chunks and their embeddings
class SimpleVectorStore:
    def __init__(self):
        self.documents, self.embeddings, self.metadatas, self.ids = [], [], [], []

    def add(self, e, d, m, i):
        self.embeddings.extend(e); self.documents.extend(d)
        self.metadatas.extend(m); self.ids.extend(i)

    def query(self, q_emb, n=3):
        if not self.embeddings:
            return {'ids':[[]],'documents':[[]],'metadatas':[[]],'distances':[[]]}
        sims = cosine_similarity(q_emb, self.embeddings)[0]
        top  = np.argsort(sims)[::-1][:n]
        return {
            'ids':       [[self.ids[i]       for i in top]],
            'documents': [[self.documents[i] for i in top]],
            'metadatas': [[self.metadatas[i] for i in top]],
            'distances': [[1 - sims[i]       for i in top]]
        }

    def count(self): return len(self.documents)

# Main class that loads papers and answers questions using RAG
class BlueberryRAG:
    def __init__(self):
        self.tfidf = TfidfVectorizer(max_features=3000, stop_words="english")
        self.fitted = False
        self.collection = SimpleVectorStore()

    def _prep(self, t):
        return re.sub(r"\s+", " ", t or "").strip().lower()

    def load_papers_from_files(self, papers):
        docs, metas, ids = [], [], []
        for p in papers:
            try:
                fn = f"paper_{p['doc_id']}.pdf"
                download_pdf(p['pdf_url'], fn)
                full = extract_text_from_pdf(fn)
                chunks = split_text(full)
                for i, chunk in enumerate(chunks):
                    if is_garbage_chunk(chunk):
                        continue
                    docs.append(self._prep(chunk))
                    metas.append({
                        "doc_id":   str(p['doc_id']),
                        "title":    p['title'],
                        "authors":  p['authors'],
                        "year":     str(p['year']),
                        "doi":      p['doi'],
                        "chunk_id": str(i)
                    })
                    ids.append(f"doc_{p['doc_id']}_chunk_{i}")
                print(f"Loaded paper {p['doc_id']} ({len(chunks)} chunks)")
            except Exception as e:
                print(f"Could not load paper {p['doc_id']}: {e}")
        if docs:
            embeddings = self.tfidf.fit_transform(docs).toarray()
            self.fitted = True
            self.collection.add(embeddings, docs, metas, ids)
        print(f"Total chunks: {self.collection.count()}")

    def query(self, question, n_results=3):
        if self.collection.count() == 0:
            return {"response": "No papers loaded.", "papers_found": 0}
        q_emb = self.tfidf.transform([self._prep(question)]).toarray()
        # Fetch more candidates so we can pick 1 best chunk per paper
        candidates = self.collection.query(q_emb, min(n_results * 10, self.collection.count()))
        all_docs   = candidates['documents'][0]
        all_metas  = candidates['metadatas'][0]
        all_dists  = candidates['distances'][0]
        # Diversify: keep only the best chunk per paper (doc_id)
        seen, d_docs, d_metas, d_dists = set(), [], [], []
        for doc, meta, dist in zip(all_docs, all_metas, all_dists):
            pid = meta['doc_id']
            if pid not in seen:
                seen.add(pid)
                d_docs.append(doc); d_metas.append(meta); d_dists.append(dist)
            if len(d_docs) >= n_results:
                break
        docs, metas, dists = d_docs, d_metas, d_dists
        res = {'documents': [docs], 'metadatas': [metas], 'distances': [dists]}
        context = ""
        for doc, meta, dist in zip(docs, metas, dists):
            context += (
                f"\nSource: {meta['title']} ({meta['year']})\n"
                f"Authors: {meta['authors']}  Relevance: {round((1-dist)*100,1)}%\n"
                f"Text: {doc[:500]}\n"
            )
        prompt = (
            "You are an AI assistant for a blueberry plant monitoring system.\n"
            "Answer ONLY using the academic excerpts below. Be practical and farmer-friendly.\n"
            f"Question: {question}\nExcerpts: {context}\nWrite a clear, concise answer."
        )
        try:
            answer = client.models.generate_content(
                model="gemini-2.5-flash", contents=prompt).text
        except Exception as e:
            answer = f"LLM error: {e}"
        return {"response": answer, "papers_found": len(docs), "raw": res}

# Paper metadata used by the RAG system
BLUEBERRY_PAPERS = [
    {"doc_id":1,"title":"Managing fruit rot diseases of Vaccinium corymbosum",
     "authors":"Neugebauer et al.","journal":"Frontiers in Plant Science","year":2024,
     "doi":"https://doi.org/10.3389/fpls.2024.1428769",
     "pdf_url":"https://www.frontiersin.org/articles/10.3389/fpls.2024.1428769/pdf"},
    {"doc_id":2,"title":"Blueberry Culture and Pest, Disease, and Abiotic Disorder Management",
     "authors":"Fulcher et al.","journal":"HortTechnology","year":2015,
     "doi":"https://doi.org/10.24266/0738-2898-33.1.33",
     "pdf_url":"https://journals.ashs.org/horttech/downloadpdf/journals/horttech/25/5/article-p587.pdf"},
    {"doc_id":3,"title":"Highbush Blueberry: Cultivation, Protection, Breeding and Biotechnology",
     "authors":"Prodorutti et al.","journal":"EJPSB","year":2007,
     "doi":"http://www.globalsciencebooks.info/Online/GSBOnline/images/0706/EJPSB_1(1)/EJPSB_1(1)44-56o.pdf",
     "pdf_url":"http://www.globalsciencebooks.info/Online/GSBOnline/images/0706/EJPSB_1(1)/EJPSB_1(1)44-56o.pdf"},
    {"doc_id":4,"title":"Effects of NPK fertilization on yield and berry quality of blueberry",
     "authors":"Zhang et al.","journal":"PLOS ONE","year":2023,
     "doi":"https://doi.org/10.1371/journal.pone.0283137",
     "pdf_url":"https://journals.plos.org/plosone/article/file?id=10.1371/journal.pone.0283137&type=printable"},
    {"doc_id":5,"title":"Breeding blueberries for a changing global environment",
     "authors":"Lobos & Hancock","journal":"Frontiers in Plant Science","year":2015,
     "doi":"https://doi.org/10.3389/fpls.2015.00782",
     "pdf_url":"https://www.frontiersin.org/articles/10.3389/fpls.2015.00782/pdf"}
]

# Create and load the RAG system
rag_system = BlueberryRAG()
rag_system.load_papers_from_files(BLUEBERRY_PAPERS)
print("RAG system ready!")


Loaded paper 1 (149 chunks)
Loaded paper 2 (1 chunks)
Loaded paper 3 (129 chunks)
Loaded paper 4 (52 chunks)
Loaded paper 5 (111 chunks)
Total chunks: 437
RAG system ready!


## AI Agent: PlantHealthAgent

In this part we create an AI agent that checks the plant condition.
The agent reads sensor data and the latest image result, decides the urgency level, and then asks Gemini to generate an action plan.

In [13]:
# AI agent that combines sensor data and image diagnosis
# to create a plant health action plan

class PlantHealthAgent:
    # Checks plant health using sensors, image history, and Gemini

    def __init__(self, gemini_client):
        self._client = gemini_client
        self.steps   = []

    # Observe: read sensor data
    def _observe_sensors(self):
        readings = {}
        for feed in ['temperature', 'humidity', 'soil']:
            df, _, from_cache = fetch_sensor_data(feed, 5)
            if df is not None:
                val = round(float(df['value'].iloc[0]), 2)
                _, status, _, unit = evaluate_sensor_status(feed, val)
                readings[feed] = {
                    'value': val, 'unit': unit,
                    'status': status, 'from_cache': from_cache
                }
        self.steps.append(f"[OBSERVE] Sensors: {list(readings.keys())}")
        return readings

    # Observe: read the latest image diagnosis
    def _observe_image(self):
        try:
            data = FBconn.get(IMAGES_PATH, None)
            if data and isinstance(data, dict):
                items = sorted(data.values(),
                               key=lambda x: x.get('uploaded_at', ''), reverse=True)
                if items:
                    img = items[0]
                    self.steps.append(
                        "[OBSERVE] Image: "
                        + img.get('prediction', 'N/A')
                        + f" ({img.get('confidence', 0)}%)"
                    )
                    return img
        except Exception:
            pass
        self.steps.append("[OBSERVE] No image history found")
        return None

    # Reason: decide the urgency level
    def _reason(self, sensors, image):
        critical = [f for f, d in sensors.items() if d['status'] == 'Critical']
        warning  = [f for f, d in sensors.items() if d['status'] == 'Warning']
        urgency  = 'high' if critical else ('medium' if warning else 'low')
        r = {
            'critical': critical, 'warning': warning, 'urgency': urgency,
            'image_pred': image.get('prediction', 'No diagnosis') if image else 'No images',
            'image_conf': image.get('confidence', 0)             if image else 0
        }
        self.steps.append(f"[REASON] urgency={urgency}  critical={critical}")
        return r

    # Act: generate a recommended action plan
    def _act(self, sensors, reasoning):
        lines = "\n".join(
            f"  - {f.capitalize()}: {d['value']} {d['unit']} ({d['status']})"
            for f, d in sensors.items()
        )
        prompt = (
            "You are a precision-agriculture AI agent for a blueberry monitoring system.\n\n"
            f"SENSOR READINGS:\n{lines}\n\n"
            f"LATEST IMAGE DIAGNOSIS: {reasoning['image_pred']} "
            f"(confidence: {reasoning['image_conf']}%)\n"
            f"CRITICAL SENSORS: {', '.join(reasoning['critical']) or 'None'}\n"
            f"WARNINGS:         {', '.join(reasoning['warning'])  or 'None'}\n\n"
            "Provide:\n"
            "1. Overall health status (1 sentence, emoji prefix)\n"
            "2. Most urgent action RIGHT NOW (1 sentence)\n"
            "3. Next 24-hour actions (2 bullet points)\n"
            "4. Sensor to prioritise next time\n\n"
            "Be specific, concise, and farmer-friendly."
        )
        try:
            plan = self._client.models.generate_content(
                model="gemini-2.5-flash", contents=prompt).text
        except Exception:
            msgs = {
                'high':   "Critical: immediate action needed for "
                          + ", ".join(reasoning['critical']),
                'medium': "Monitor closely: "
                          + ", ".join(reasoning['warning']),
                'low':    "Plant healthy - maintain current routine."
            }
            plan = msgs.get(reasoning['urgency'], "Check sensors manually.")
        self.steps.append("[ACT] Action plan generated via Gemini")
        return plan

    # Runs the full agent flow
    def run(self):
        self.steps = []
        sensors = self._observe_sensors()
        image   = self._observe_image()
        reason  = self._reason(sensors, image)
        plan    = self._act(sensors, reason)
        return {
            'sensors': sensors, 'reasoning': reason,
            'action_plan': plan, 'steps': self.steps
        }

# Create the plant health agent
plant_agent = PlantHealthAgent(client)
print("PlantHealthAgent ready")


PlantHealthAgent ready


## Sensor Functions: Firebase Fallback

In this part we get sensor readings for temperature, humidity, and soil moisture.
If the live server does not work, the system tries Firebase cache, and if there is no cache it shows demo data.

In [14]:
# Creates demo sensor data if live and cached data are not available
def _demo_sensor_data(feed):
    import numpy as np
    now = datetime.now()
    base = {"temperature": 22.0, "humidity": 62.0, "soil": 58.0}.get(feed, 50.0)
    rows = []
    for i in range(20):
        t = now - timedelta(minutes=i*30)
        v = round(base + np.random.uniform(-2, 2), 2)
        rows.append({"created_at": t, "value": v})
    df = pd.DataFrame(rows)
    df["created_at"] = pd.to_datetime(df["created_at"])
    df["value"]      = pd.to_numeric(df["value"], errors="coerce")
    return df

# Saves recent sensor readings to Firebase
def _save_sensor_cache(feed, df):
    try:
        records = df.head(30).copy()
        records["created_at"] = records["created_at"].dt.strftime("%Y-%m-%d %H:%M:%S")
        FBconn.put(SENSOR_CACHE_PATH, feed, {
            "data":     records.to_dict("records"),
            "saved_at": datetime.now().strftime("%Y-%m-%d %H:%M")
        })
    except Exception as e:
        print(f"  [cache save] {e}")

# Loads saved sensor readings from Firebase
def _load_sensor_cache(feed):
    try:
        result = FBconn.get(SENSOR_CACHE_PATH, feed)
        if result and "data" in result:
            df = pd.DataFrame(result["data"])
            df["created_at"] = pd.to_datetime(df["created_at"])
            df["value"]      = pd.to_numeric(df["value"], errors="coerce")
            return df, result.get("saved_at", "unknown time")
    except Exception:
        pass
    return None, None

# Gets sensor data: first from live server, then Firebase cache, then demo data
def fetch_sensor_data(feed, limit):
    # 1) Try live server
    try:
        records = SensorMicroservice.get_history(feed, limit)
        df = pd.DataFrame(records)
        df["created_at"] = pd.to_datetime(df["created_at"])
        df["value"]      = pd.to_numeric(df["value"], errors="coerce")
        _save_sensor_cache(feed, df)
        return df, "OK", "live"
    except Exception as live_err:
        print(f"  [sensor live] {live_err}")

    # 2) Try Firebase cache
    df_cache, saved_at = _load_sensor_cache(feed)
    if df_cache is not None:
        return df_cache, f"Cached data (saved: {saved_at})", "cached"

    # 3) Demo data fallback so the UI never breaks
    df_demo = _demo_sensor_data(feed)
    return df_demo, "Demo data (server unreachable, no cache yet)", "demo"

# Checks if the sensor value is healthy, warning, or critical
def evaluate_sensor_status(feed, value):
    if feed == "temperature":
        u = "C"
        if 18 <= value <= 26:    return "OK","Healthy","Temperature is ideal for blueberries.",u
        elif 15 <= value <= 30:  return "W","Warning","Temperature slightly outside ideal range.",u
        else:                    return "C","Critical","Temperature may stress the plant.",u
    if feed == "humidity":
        u = "%"
        if 50 <= value <= 70:    return "OK","Healthy","Humidity is suitable for blueberries.",u
        elif 40 <= value <= 80:  return "W","Warning","Humidity not ideal. Monitor closely.",u
        else:                    return "C","Critical","Humidity may increase disease risk.",u
    if feed == "soil":
        u = "%"
        if 50 <= value <= 70:    return "OK","Healthy","Soil moisture is balanced.",u
        elif 35 <= value <= 85:  return "W","Warning","Soil moisture not ideal. Check drainage.",u
        else:                    return "C","Critical","Soil moisture risky - check immediately.",u
    return "?","Unknown","No data available.",""

# Returns an icon based on the sensor status
def _status_icon(code):
    return {"OK": "\U0001f7e2", "W": "\U0001f7e1", "C": "\U0001f534"}.get(code, "⚪")

# Builds the sensor screen output: status card, graph, and table
def get_sensor_history(feed, limit):
    df, msg, mode = fetch_sensor_data(feed, int(limit))

    if mode == "cached":
        banner = (
            "<div style='background:#FFF3CD;border:1px solid #FFC107;"
            "border-radius:14px;padding:12px 16px;margin-bottom:14px;"
            "color:#856404;font-weight:700;'>"
            "\u26a0\ufe0f Live sensor server unavailable. "
            "Showing <strong>last saved data</strong> from Firebase. "
            "This data may not reflect current conditions.</div>"
        )
    elif mode == "demo":
        banner = (
            "<div style='background:#FFF3CD;border:1px solid #FFC107;"
            "border-radius:14px;padding:12px 16px;margin-bottom:14px;"
            "color:#856404;font-weight:700;'>"
            "\u26a0\ufe0f Sensor server is waking up (Render cold start). "
            "Showing <strong>demo data</strong>. "
            "Please wait ~60 seconds and refresh to get live readings.</div>"
        )
    else:
        banner = ""

    v    = df["value"].iloc[0]
    t    = df["created_at"].iloc[0].strftime("%Y-%m-%d %H:%M")
    code, status, rec, unit = evaluate_sensor_status(feed, v)
    icon = _status_icon(code)
    cls  = {"Healthy": "healthy-card", "Warning": "warning-card"}.get(status, "critical-card")

    suffix_label = {"cached": " (Cached)", "demo": " (Demo)"}.get(mode, "")
    html = (
        banner +
        f"<div class=\'status-card {cls}\'>"
        f"<div class=\'status-icon\'>{icon}</div>"
        f"<div class=\'status-title\'>{status}</div>"
        f"<div class=\'status-value\'>{v} {unit}</div>"
        f"<div class=\'status-subtitle\'>Current {feed.capitalize()} Reading</div>"
        f"<div class=\'status-row\'><span>Last Update</span><strong>{t}</strong></div>"
        f"<div class=\'status-row\'><span>Samples</span><strong>{len(df)}</strong></div>"
        f"<div class=\'recommendation-box\'>{rec}</div></div>"
    )

    fig, ax = plt.subplots(figsize=(9, 4.5))
    fig.patch.set_facecolor("#faf7ff"); ax.set_facecolor("#ffffff")
    ax.plot(df["created_at"], df["value"], marker="o", linewidth=2.5, color="#5b3cc4",
            markerfacecolor="#2f195f", markeredgecolor="white", markeredgewidth=1.5)
    ax.fill_between(df["created_at"], df["value"], alpha=0.15, color="#5b3cc4")
    ax.set_title(f"{feed.capitalize()} Trend{suffix_label}", fontsize=14, fontweight="bold")
    ax.set_xlabel("Time"); ax.set_ylabel(f"Value ({unit})")
    ax.grid(True, alpha=0.3)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.xticks(rotation=35); plt.tight_layout()

    tbl = df[["created_at", "value"]].copy()
    tbl["created_at"] = tbl["created_at"].dt.strftime("%d/%m/%Y %H:%M")
    tbl.columns = ["Date & Time", f"Value ({unit})"]

    return html, fig, tbl

print("Sensor functions ready (live > Firebase cache > demo fallback)")


Sensor functions ready (live > Firebase cache > demo fallback)


## Dashboard + Gamification

In this part we build the dashboard screen and the gamification system.
The dashboard shows sensor cards, graphs, health score, AI agent analysis, and user progress.

In [15]:
# Levels and achievements used in the gamification system
LEVELS = [
    {'min':    0, 'max':  100, 'name': 'Seedling', 'icon': '\U0001f331'},
    {'min':  100, 'max':  300, 'name': 'Sprout',   'icon': '\U0001f33f'},
    {'min':  300, 'max':  600, 'name': 'Plant',    'icon': '\U0001fab4'},
    {'min':  600, 'max': 1200, 'name': 'Grower',   'icon': '\U0001f9d1‍\U0001f33e'},
    {'min': 1200, 'max': None, 'name': 'Expert',   'icon': '\U0001f3c6'},
]

ACHIEVEMENTS = {
    'first_check':  ('First Check!',       '\U0001f31f', 'Loaded your first sensor reading'),
    'ten_checks':   ('Sensor Scout',       '\U0001f52d', 'Checked sensors 10+ times'),
    'first_upload': ('Plant Photographer', '\U0001f4f8', 'Uploaded your first plant photo'),
    'all_healthy':  ('All Green!',         '\U0001f49a', 'All sensors in Healthy range'),
    'streak_3':     ('3-Day Streak',       '\U0001f525', 'Dashboard 3 days in a row'),
    'streak_7':     ('Weekly Watcher',     '\U0001f308', '7-day consecutive streak'),
}

# Finds the user level according to XP
def _get_level(xp):
    for i, lv in enumerate(LEVELS):
        if lv['max'] is None or xp < lv['max']:
            lo, hi = lv['min'], lv['max']
            pct = round((xp - lo) / (hi - lo) * 100) if hi else 100
            return i, lv['name'], lv['icon'], min(100, pct), hi
    return 4, 'Expert', '\U0001f3c6', 100, None

# Loads the gamification state from Firebase
def get_gamification_state():
    try:
        s = FBconn.get(GAMIFICATION_PATH, None)
        if s and isinstance(s, dict):
            return s
    except Exception:
        pass
    return {'xp': 0, 'checks': 0, 'uploads': 0, 'streak': 0,
            'last_date': '', 'earned': []}

# Saves the gamification state to Firebase
def save_gamification_state(s):
    try:
        FBconn.put(GAMIFICATION_PATH, None, s)
    except Exception:
        pass

# Updates XP, streaks, and achievements after user actions
def update_gamification(action='dashboard_check', all_healthy=False):
    XP_MAP = {'dashboard_check': 15, 'sensor_check': 10,
              'image_upload': 20, 'search': 8}
    s      = get_gamification_state()
    earned = list(s.get('earned') or [])
    today  = datetime.now().strftime('%Y-%m-%d')
    yest   = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')
    xp     = XP_MAP.get(action, 5)

    if s.get('last_date') == today:
        xp = max(2, xp // 3)
    elif s.get('last_date') == yest:
        s['streak'] = s.get('streak', 0) + 1
        xp += s['streak'] * 3
    else:
        s['streak'] = 1

    s['last_date'] = today
    s['xp']      = s.get('xp', 0) + xp
    s['checks']  = s.get('checks', 0) + 1
    if action == 'image_upload':
        s['uploads'] = s.get('uploads', 0) + 1

    new_ach = []
    def _ach(key):
        if key not in earned:
            earned.append(key)
            new_ach.append(ACHIEVEMENTS[key])

    if s['checks'] >= 1:  _ach('first_check')
    if s['checks'] >= 10: _ach('ten_checks')
    if s.get('uploads', 0) >= 1: _ach('first_upload')
    if all_healthy:       _ach('all_healthy')
    if s['streak'] >= 3:  _ach('streak_3')
    if s['streak'] >= 7:  _ach('streak_7')

    s['earned'] = earned
    save_gamification_state(s)
    return s, xp, new_ach

# Builds the gamification HTML section
def render_gamification(state, xp_gain=0, new_ach=None):
    xp     = state.get('xp', 0)
    streak = state.get('streak', 0)
    checks = state.get('checks', 0)
    earned = list(state.get('earned') or [])
    lvl_i, lvl_name, lvl_icon, pct, next_xp = _get_level(xp)

    badges = ''.join(
        f"<span title='{ACHIEVEMENTS[k][2]}' style='font-size:24px;cursor:help;'>"
        f"{ACHIEVEMENTS[k][1]}</span> "
        for k in earned if k in ACHIEVEMENTS
    )
    new_html = ''.join(
        f"<div style='background:#D1FAE5;border-radius:10px;padding:9px 13px;"
        f"margin:5px 0;color:#065F46;font-weight:700;'>"
        f"New Achievement: {icon} {name} - {desc}</div>"
        for name, icon, desc in (new_ach or [])
    )
    next_label = f"/ {next_xp} XP to next level" if next_xp else "- Max Level!"

    how_to = (
        "<div style='margin-top:20px;background:#FFFFFF;border-radius:16px;"
        "border:1px solid #E6DCF8;padding:18px;'>"
        "<div style='font-size:15px;font-weight:800;color:#2F195F;margin-bottom:12px;'>"
        "\U0001f4a1 How to earn XP & level up</div>"
        "<div style='display:grid;grid-template-columns:1fr 1fr;gap:10px;font-size:13px;'>"
        "<div style='background:#F3EEFF;border-radius:12px;padding:10px 14px;'>"
        "<b>\U0001f4f7 Upload a photo</b><br>+20 XP per upload</div>"
        "<div style='background:#F3EEFF;border-radius:12px;padding:10px 14px;'>"
        "<b>\U0001f4e1 Check sensors</b><br>+10 XP per check</div>"
        "<div style='background:#F3EEFF;border-radius:12px;padding:10px 14px;'>"
        "<b>\U0001f50d Search papers</b><br>+8 XP per search</div>"
        "<div style='background:#F3EEFF;border-radius:12px;padding:10px 14px;'>"
        "<b>\U0001f4ca Open dashboard</b><br>+15 XP per visit</div>"
        "</div>"
        "<div style='margin-top:14px;font-size:13px;font-weight:800;color:#2F195F;margin-bottom:8px;'>"
        "\U0001f3c6 Levels</div>"
        "<div style='display:flex;flex-wrap:wrap;gap:8px;'>"
        "<span style='background:#EDE9FE;color:#5B3CC4;padding:4px 12px;border-radius:20px;font-size:12px;font-weight:700;'>\U0001f331 Seedling 0&ndash;100 XP</span>"
        "<span style='background:#EDE9FE;color:#5B3CC4;padding:4px 12px;border-radius:20px;font-size:12px;font-weight:700;'>\U0001f33f Sprout 100&ndash;300 XP</span>"
        "<span style='background:#EDE9FE;color:#5B3CC4;padding:4px 12px;border-radius:20px;font-size:12px;font-weight:700;'>\U0001fab4 Plant 300&ndash;600 XP</span>"
        "<span style='background:#EDE9FE;color:#5B3CC4;padding:4px 12px;border-radius:20px;font-size:12px;font-weight:700;'>\U0001f9d1 Grower 600&ndash;1200 XP</span>"
        "<span style='background:#EDE9FE;color:#5B3CC4;padding:4px 12px;border-radius:20px;font-size:12px;font-weight:700;'>\U0001f3c6 Expert 1200+ XP</span>"
        "</div>"
        "<div style='margin-top:14px;font-size:13px;font-weight:800;color:#2F195F;margin-bottom:8px;'>"
        "\U0001f525 Streaks &amp; Achievements</div>"
        "<div style='font-size:12px;color:#6B5A86;line-height:1.7;'>"
        "Use the app on consecutive days to build your streak &mdash; streak bonus adds extra XP each day.<br>"
        "Unlock achievements by reaching milestones: first photo, 10 sensor checks, 3-day streak, and more."
        "</div></div>"
    )

    return (
        "<div class='gamification-box' style='padding:22px;border-radius:22px;"
        "background:#F3EEFF;border:1px solid #DDD3F3;'>"
        f"<div style='display:flex;align-items:center;gap:14px;margin-bottom:14px;'>"
        f"<span style='font-size:40px;'>{lvl_icon}</span>"
        f"<div><div style='font-size:20px;font-weight:900;color:#2F195F;'>"
        f"Level {lvl_i+1}: {lvl_name}</div>"
        f"<div style='color:#6B5A86;font-size:13px;'>+{xp_gain} XP earned this session</div>"
        "</div></div>"
        "<div style='background:#E5E7EB;border-radius:20px;height:13px;overflow:hidden;margin:6px 0;'>"
        f"<div style='background:linear-gradient(90deg,#5B3CC4,#7C5CE6);height:100%;"
        f"width:{pct}%;border-radius:20px;transition:width 1.2s ease;'></div></div>"
        f"<div style='text-align:right;font-size:12px;color:#6B5A86;margin-bottom:10px;'>"
        f"{xp} XP {next_label}</div>"
        "<div style='display:flex;gap:24px;flex-wrap:wrap;'>"
        f"<div style='text-align:center;'><div style='font-size:26px;font-weight:900;color:#5B3CC4;'>"
        f"{streak}\U0001f525</div><div style='font-size:12px;color:#6B5A86;'>Day Streak</div></div>"
        f"<div style='text-align:center;'><div style='font-size:26px;font-weight:900;color:#5B3CC4;'>"
        f"{checks}</div><div style='font-size:12px;color:#6B5A86;'>Total Checks</div></div>"
        f"<div style='text-align:center;'><div style='font-size:26px;font-weight:900;color:#5B3CC4;'>"
        f"{len(earned)}</div><div style='font-size:12px;color:#6B5A86;'>Achievements</div></div>"
        "</div>"
        + (f"<div style='margin-top:12px;letter-spacing:4px;'>{badges}</div>" if badges else "")
        + new_html
        + "</div>"
        + how_to
    )

# Loads the latest sensor data for the dashboard
def fetch_dashboard_data(limit=30):
    out = {}
    for f in ['temperature', 'humidity', 'soil']:
        df, _, _ = fetch_sensor_data(f, limit)
        if df is not None and len(df) > 0:
            out[f] = df
    return out

# Calculates the overall plant health score
def calculate_health_score(data):
    scores = []
    for feed, df in data.items():
        _, status, _, _ = evaluate_sensor_status(feed, df['value'].iloc[0])
        scores.append(100 if status == 'Healthy' else 65 if status == 'Warning' else 30)
    return round(sum(scores) / len(scores)) if scores else 0

# Builds the sensor summary cards for the dashboard
def create_dashboard_cards(data):
    score = calculate_health_score(data)
    t = data.get('temperature', pd.DataFrame([{'value': 0}]))['value'].iloc[0]
    h = data.get('humidity',    pd.DataFrame([{'value': 0}]))['value'].iloc[0]
    s = data.get('soil',        pd.DataFrame([{'value': 0}]))['value'].iloc[0]
    return (
        "<div class='dashboard-grid'>"
        f"<div class='dashboard-card'><div class='dashboard-icon'>\U0001f321</div>"
        f"<div class='dashboard-title'>Temperature</div>"
        f"<div class='dashboard-value'>{t} C</div>"
        f"<div class='dashboard-subtitle'>Latest reading</div></div>"
        f"<div class='dashboard-card'><div class='dashboard-icon'>\U0001f4a7</div>"
        f"<div class='dashboard-title'>Humidity</div>"
        f"<div class='dashboard-value'>{h} %</div>"
        f"<div class='dashboard-subtitle'>Latest reading</div></div>"
        f"<div class='dashboard-card'><div class='dashboard-icon'>\U0001f331</div>"
        f"<div class='dashboard-title'>Soil Moisture</div>"
        f"<div class='dashboard-value'>{s} %</div>"
        f"<div class='dashboard-subtitle'>Latest reading</div></div>"
        f"<div class='dashboard-card health-score-card'>"
        f"<div class='dashboard-icon'>\U0001fad0</div>"
        f"<div class='dashboard-title'>Plant Health Score</div>"
        f"<div class='dashboard-value'>{score}%</div>"
        f"<div class='dashboard-subtitle'>Overall condition</div></div>"
        "</div>"
    )

# Creates the dashboard sensor comparison graph
def create_dashboard_plot(data):
    fig, ax = plt.subplots(figsize=(11, 4.5))
    fig.patch.set_facecolor('#faf7ff'); ax.set_facecolor('#ffffff')
    for feed, df in data.items():
        ax.plot(df['created_at'], df['value'], marker='o',
                linewidth=2, label=feed.capitalize())
    ax.set_title('Sensor Trends Comparison', fontsize=15, fontweight='bold')
    ax.set_xlabel('Time'); ax.set_ylabel('Value')
    ax.grid(True, alpha=0.25)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.legend(); plt.xticks(rotation=35); plt.tight_layout()
    return fig

# Loads all dashboard parts together
def load_dashboard(sel):
    lim  = {'today': 30, 'yesterday': 60, 'week': 100}.get(sel, 30)
    data = fetch_dashboard_data(lim)
    if not data:
        return '<p>Could not load sensor data.</p>', None, ''

    cards = create_dashboard_cards(data)
    plot  = create_dashboard_plot(data)
    score = calculate_health_score(data)
    all_ok = all(
        evaluate_sensor_status(f, df['value'].iloc[0])[1] == 'Healthy'
        for f, df in data.items()
    )
    state, xp_gain, new_ach = update_gamification('dashboard_check', all_ok)

    # Run the AI agent
    agent_result = plant_agent.run()
    steps_html   = ''.join(
        f"<div style='color:#6B5A86;font-size:13px;'>- {s}</div>"
        for s in agent_result['steps']
    )
    health_label = ('Excellent' if score >= 80 else
                    'Needs Monitoring' if score >= 60 else 'Needs Attention')

    summary = (
        "<div class='dashboard-recommendation'>"
        f"<h3>\U0001fad0 Plant Health Summary</h3>"
        f"<p>Health Score: <strong>{score}%</strong> - {health_label}</p>"
        "<div style='background:#F0F4FF;border-radius:22px;padding:20px;margin:14px 0;"
        "border:1px solid #C7D7FF;'>"
        "<h3 style='margin:0 0 10px;color:#2F195F;'>\U0001f916 AI Agent Analysis</h3>"
        "<div style='background:#fff;border-radius:12px;padding:10px 14px;"
        "font-size:13px;color:#6B5A86;margin-bottom:10px;'>"
        "<strong>Agent reasoning steps:</strong><br>" + steps_html + "</div>"
        "<div style='white-space:pre-wrap;color:#2F195F;line-height:1.75;'>"
        + agent_result['action_plan'] + "</div></div>"
        "<div style='margin-top:18px;'>"
        "<h3>\U0001f3c6 Your Progress</h3>"
        + render_gamification(state, xp_gain, new_ach) +
        "</div></div>"
    )
    return cards, plot, summary

print('Dashboard + gamification functions ready')


Dashboard + gamification functions ready


In [16]:
# Install libraries needed for the image diagnosis model
!pip install -q huggingface_hub tensorflow

## Plant Image Upload + AI Diagnosis + Firebase Storage

In this part the user can upload a blueberry plant image.
The system predicts the plant condition, asks Gemini for recommendations, and saves the result in Firebase.

In [17]:
# Store uploaded images for the current session
uploaded_images = []

# Load the disease classification model from Hugging Face
MODEL_REPO_ID = "MaayanSal20/blueberry-disease-classifier"
model_path    = hf_hub_download(repo_id=MODEL_REPO_ID,
                                filename="blueberry_disease_classifier.keras")
classes_path  = hf_hub_download(repo_id=MODEL_REPO_ID,
                                filename="class_names.json")
plant_model   = tf.keras.models.load_model(model_path)

with open(classes_path, "r") as f:
    class_names = json.load(f)

print("HuggingFace disease model loaded:", MODEL_REPO_ID)

# Converts an image to base64 before saving it to Firebase
def _img_to_b64(image_path, size=(300, 300), quality=50):
    if image_path is None:
        return None
    img = Image.open(image_path).convert("RGB")
    img.thumbnail(size)
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality)
    return base64.b64encode(buf.getvalue()).decode("utf-8")

# Saves image diagnosis data to Firebase
def _save_image_to_firebase(entry):
    try:
        FBconn.post(IMAGES_PATH, {
            'plant_name':    entry['plant_name'],
            'health_status': entry['health_status'],
            'prediction':    entry.get('prediction', ''),
            'confidence':    entry.get('confidence', 0),
            'notes':         entry.get('notes', ''),
            'uploaded_at':   entry['uploaded_at'],
            'thumbnail':     _img_to_b64(entry['image'])
        })
    except Exception as e:
        print(f"  [Firebase image] {e}")

# Predicts the disease class from the uploaded image
def predict_plant_condition(image_path):
    img  = Image.open(image_path).convert("RGB").resize((224, 224))
    arr  = np.expand_dims(np.array(img), axis=0)
    pred = plant_model.predict(arr, verbose=0)[0]
    idx  = np.argmax(pred)
    return class_names[idx], float(pred[idx])

# Converts the prediction into a general health status
def get_status_from_prediction(pred_class, confidence):
    text = pred_class.lower()
    if confidence < 0.50:
        return "Low Confidence"
    if "healthy" in text:
        return "Healthy"
    if any(w in text for w in ["critical", "severe", "disease", "rot", "botrytis", "anthracnose"]):
        return "Critical"
    return "Needs Attention"

# Uses Gemini to generate simple recommendations for the user
def generate_gemini_recommendation(predicted_class, confidence, notes):
    pct   = confidence * 100
    level = ("High confidence" if pct >= 70
             else "Medium confidence" if pct >= 50 else "Low confidence")
    color = "#22C55E" if pct >= 70 else "#F59E0B" if pct >= 50 else "#EF4444"
    icon  = "\U0001f7e2" if pct >= 70 else "\U0001f7e1" if pct >= 50 else "\U0001f534"

    prompt = (
        "You are an AI assistant for a blueberry plant monitoring system.\n"
        f"Model prediction: {predicted_class}\n"
        f"Confidence: {pct:.2f}%\n"
        f"User notes: {notes or 'None'}\n\n"
        "Return ONLY a valid JSON object with keys:\n"
        "condition (max 4 words), meaning (1 short sentence), "
        "actions (exactly 3 short strings).\nNo markdown, no extra text."
    )
    try:
        text = client.models.generate_content(
            model="gemini-2.5-flash", contents=prompt).text
        text = text.replace("```json", "").replace("```", "").strip()
        data = json.loads(text)
        condition = data.get("condition", predicted_class)
        meaning   = data.get("meaning", "AI detected a possible plant condition.")
        actions   = data.get("actions", [])
    except Exception:
        condition = predicted_class
        meaning   = "Gemini unavailable - check the plant manually."
        actions   = ["Upload a clearer photo", "Inspect the plant closely", "Try again later"]

    note = (
        "<div style='margin-top:12px;padding:12px;border-radius:14px;"
        "background:#FFF7ED;color:#9A3412;'>"
        "Low confidence - upload a clearer photo with good lighting.</div>"
    ) if pct < 50 else ""

    acts = "".join(
        f"<li style='color:#2F195F;margin-bottom:8px;'>{a}</li>"
        for a in actions[:3]
    )
    return (
        "<div style='background:#F8F6FF;border:1px solid #DDD3F3;border-radius:22px;"
        "padding:20px;color:#2F195F;box-shadow:0 8px 20px rgba(47,25,95,.08);"
        "font-family:Arial,sans-serif;'>"
        "<h2 style='margin-top:0;color:#2F195F;'>\U0001fad0 Plant Diagnosis</h2>"
        f"<div style='font-size:20px;font-weight:800;color:{color};'>{icon} {condition}</div>"
        f"<p><strong>Confidence:</strong> {pct:.2f}% - {level}</p>"
        f"<p><strong>Meaning:</strong> {meaning}</p>"
        "<div><strong>Recommended actions:</strong>"
        f"<ul style='padding-left:22px;margin-top:8px;'>{acts}</ul></div>"
        + note + "</div>"
    )


# Handles the full image upload process
def upload_plant_image(image, plant_name, notes):
    if image is None:
        return "Please upload an image.", build_image_gallery(), "", load_image_log()
    if not (plant_name or "").strip():
        plant_name = "Blueberry Plant"

    try:
        pred_class, conf = predict_plant_condition(image)
        status           = get_status_from_prediction(pred_class, conf)
        ai_result        = generate_gemini_recommendation(pred_class, conf, notes)
    except Exception as e:
        pred_class, conf = "Not detected", 0.0
        status           = "Not checked"
        ai_result        = (
            "<div style='background:#FFF0F0;border-radius:16px;"
            f"padding:16px;color:#991B1B;'>Could not analyse: {e}</div>"
        )

    entry = {
        'image':         image,
        'plant_name':    plant_name.strip(),
        'health_status': status,
        'prediction':    pred_class,
        'confidence':    round(conf * 100, 2),
        'notes':         (notes or "").strip(),
        'ai_analysis':   ai_result,
        'uploaded_at':   datetime.now().strftime('%Y-%m-%d %H:%M')
    }
    uploaded_images.append(entry)
    _save_image_to_firebase(entry)
    update_gamification('image_upload')

    return (
        f"Analysed '{plant_name}' - {pred_class} ({conf*100:.1f}%)",
        build_image_gallery(),
        ai_result,
        load_image_log()
    )

# Builds the gallery of images uploaded in this session
def build_image_gallery():
    return [
        (e['image'],
         f"{e['plant_name']} - {e['prediction']} ({e['confidence']}%) | {e['uploaded_at']}")
        for e in uploaded_images
    ]

# Loads image upload history from Firebase
def load_image_log():
    try:
        data = FBconn.get(IMAGES_PATH, None)
        if data and isinstance(data, dict):
            items = sorted(data.values(),
                           key=lambda x: x.get('uploaded_at', ''), reverse=True)
            return pd.DataFrame([{
                'Plant':       e.get('plant_name', ''),
                'AI Status':   e.get('health_status', ''),
                'Prediction':  e.get('prediction', ''),
                'Confidence':  f"{e.get('confidence', 0)}%",
                'Notes':       str(e.get('notes', ''))[:50],
                'Uploaded At': e.get('uploaded_at', '')
            } for e in items])
    except Exception:
        pass
    if not uploaded_images:
        return pd.DataFrame(columns=['Plant', 'AI Status', 'Prediction',
                                     'Confidence', 'Notes', 'Uploaded At'])
    return pd.DataFrame([{
        'Plant':       e['plant_name'],
        'AI Status':   e['health_status'],
        'Prediction':  e.get('prediction', ''),
        'Confidence':  f"{e.get('confidence', 0)}%",
        'Notes':       e['notes'][:50],
        'Uploaded At': e['uploaded_at']
    } for e in uploaded_images])

print("Image upload + Firebase storage functions ready")


HuggingFace disease model loaded: MaayanSal20/blueberry-disease-classifier
Image upload + Firebase storage functions ready


## Search / RAG Handler

In this part we handle the user search query.
The system retrieves relevant paper chunks, shows the source papers, and creates a short AI summary based on the results.

In [18]:
# Links to the full papers shown in the search results
PAPER_URLS = {
    '1': 'https://doi.org/10.3389/fpls.2024.1428769',
    '2': 'https://doi.org/10.24266/0738-2898-33.1.33',
    '3': 'http://www.globalsciencebooks.info/Online/GSBOnline/images/0706/EJPSB_1(1)/EJPSB_1(1)44-56o.pdf',
    '4': 'https://doi.org/10.1371/journal.pone.0283137',
    '5': 'https://doi.org/10.3389/fpls.2015.00782'
}

# Runs the RAG search and builds the HTML result
def run_search(query, n_results):
    if not query.strip():
        return '<p>Please enter a search query.</p>'
    result = rag_system.query(query, n_results=int(n_results))
    ai_ans = result.get('response', '')
    raw    = result.get('raw', {})

    if not raw or not raw.get('documents', [[]])[0]:
        return (
            "<div class='search-header'>"
            "<h2>\U0001f9e0 AI Answer</h2>"
            f"<p>{ai_ans}</p></div>"
        )

    metas = raw['metadatas'][0]
    dists = raw['distances'][0]
    docs  = raw['documents'][0]

    # Source paper cards
    section_title = (
        "<div style='font-size:18px;font-weight:800;color:#2F195F;"
        "margin:0 0 14px;padding-bottom:8px;border-bottom:2px solid #E6DCF8;'>"
        "\U0001f4da Relevant Paper Excerpts</div>"
    )
    cards = []
    for meta, dist, doc in zip(metas, dists, docs):
        sim = round((1 - dist) * 100, 1)
        bar = chr(0x2588) * int(sim / 5) + chr(0x2591) * (20 - int(sim / 5))
        url = PAPER_URLS.get(str(meta['doc_id']), '#')
        ex  = doc[:350].replace('<', '&lt;').replace('>', '&gt;')
        cards.append(
            "<div class='search-result-card'>"
            "<div class='search-result-header'>"
            f"<span class='doc-id'>\U0001f4c4 {meta['title'][:55]}...</span>"
            f"<span class='relevance-score'>{sim}% match</span></div>"
            f"<div class='search-result-meta'>{meta['authors']} &nbsp;·&nbsp; "
            f"{meta['year']} &nbsp;·&nbsp; excerpt {int(meta['chunk_id'])+1}</div>"
            f"<div class='relevance-bar'>{bar}</div>"
            f"<div class='search-excerpt'>\"{ex}...\"</div>"
            f"<a class='doi-link' href='{url}' target='_blank'>"
            "\U0001f517 Open Full Paper</a></div>"
        )

    # AI summary based on the retrieved sources
    # Ask Gemini to generate a summary with paper citations
    context = ""
    for doc, meta, dist in zip(docs, metas, dists):
        context += (
            f"\n[Paper: {meta['title']} ({meta['year']}, {meta['authors']})]\n"
            f"Excerpt: {doc[:500]}\n"
        )
    cite_prompt = (
        "You are a research assistant for a blueberry plant monitoring system.\n"
        "Answer the question using ONLY the paper excerpts below.\n"
        "In your answer, explicitly cite the paper title in parentheses after each claim, "
        "e.g. (Neugebauer et al., 2024).\n"
        "Be concise and factual — no generic encouragement phrases.\n"
        f"Question: {query}\nPaper excerpts:\n{context}\n"
        "Write a short, cited answer (3-5 sentences)."
    )
    try:
        cited_answer = client.models.generate_content(
            model="gemini-2.5-flash", contents=cite_prompt).text
    except Exception as e:
        cited_answer = ai_ans  # fallback to original answer

    answer_html = (
        "<div class='search-header' style='margin-top:22px;'>"
        "<h2>\U0001f9e0 AI Summary</h2>"
        f"<p style='line-height:1.7;'>{cited_answer}</p></div>"
    )

    return answer_html + section_title + ''.join(cards)

print('Search/RAG handler ready')


Search/RAG handler ready


## Before & After Gallery

In this part users can save treatment cases with before and after images.
The cases are stored in Firebase and can be searched, viewed in a gallery, and used to show simple KPIs.

In [19]:
# Converts a base64 image back to a temporary image file
def _b64_to_temp(b64):
    if not b64:
        return None
    data = base64.b64decode(b64)
    path = f"{tempfile.gettempdir()}/case_{_uuid.uuid4().hex}.jpg"
    with open(path, "wb") as f:
        f.write(data)
    return path

# Builds a Firebase REST API URL
def _firebase_rest_url(path):
    return f"{FIREBASE_URL.rstrip('/')}/{path.strip('/')}.json"

# Saves a treatment case to Firebase
def _save_case(case):
    r = requests.post(_firebase_rest_url(CASES_PATH), json=case)
    r.raise_for_status()
    return r.json()

# Loads saved treatment cases from Firebase
def _load_cases():
    r = requests.get(_firebase_rest_url(CASES_PATH))
    r.raise_for_status()
    data = r.json()
    if not data:
        return []
    if isinstance(data, dict):
        cases = [dict(v, firebase_id=k)
                 for k, v in data.items() if isinstance(v, dict)]
    else:
        cases = [c for c in data if isinstance(c, dict)]
    return sorted(cases, key=lambda x: x.get('date', ''), reverse=True)

# Converts saved cases into a table
def _cases_to_df(cases):
    cols = ['Date', 'Plant Type', 'Disease / Problem',
            'Treatment', 'Result', 'Success Rating', 'Points', 'Badge']
    if not cases:
        return pd.DataFrame(columns=cols)
    return pd.DataFrame([{
        'Date':              c.get('date', ''),
        'Plant Type':        c.get('plant_type', ''),
        'Disease / Problem': c.get('disease_problem', ''),
        'Treatment':         c.get('treatment', ''),
        'Result':            c.get('result', ''),
        'Success Rating':    c.get('success_rating', ''),
        'Points':            c.get('points', ''),
        'Badge':             c.get('badge', '')
    } for c in cases], columns=cols)

# Builds the before and after image gallery
def _cases_gallery(cases):
    gallery = []
    for c in cases:
        plant = c.get('plant_type', '?')
        dis   = c.get('disease_problem', '?')
        for key, lbl in [('before_image', 'Before'), ('after_image', 'After')]:
            p = _b64_to_temp(c.get(key))
            if p:
                gallery.append((p, f"{lbl} | {plant} | {dis}"))
    return gallery

# Ranks gallery cases according to the search query
def _rank_cases(query, cases):
    query = (query or '').lower().strip()
    if not query:
        return cases
    tokens = [t for t in re.findall(r'\w+', query) if len(t) >= 3]
    ranked = []
    for c in cases:
        text = ' '.join(str(c.get(k, '')) for k in
                        ['plant_type', 'disease_problem', 'treatment', 'result']).lower()
        score = (5 if query in text else 0) + sum(1 for t in tokens if t in text)
        if score > 0:
            ranked.append((score, c))
    ranked.sort(key=lambda x: x[0], reverse=True)
    return [c for _, c in ranked] or cases

# Gives points and a badge for a saved case
def _calc_points(treatment, result):
    pts = 20
    if treatment and len(treatment.strip()) >= 30: pts += 10
    if result    and len(result.strip())    >= 20: pts += 5
    badge = ('Detailed Plant Helper' if pts >= 35
             else 'Plant Helper'     if pts >= 30 else 'New Contributor')
    return pts, badge

# Calculates KPIs for the saved cases
def _get_kpis(cases=None):
    if cases is None:
        try:   cases = _load_cases()
        except Exception: cases = []
    if not cases:
        return pd.DataFrame({
            'KPI':   ['Total cases', 'Successful (4-5)', 'Avg rating', 'Total points'],
            'Value': [0, 0, 0, 0]
        })
    ratings, points, dc = [], 0, {}
    for c in cases:
        try:    ratings.append(int(c.get('success_rating', 0)))
        except: pass
        try:    points += int(c.get('points', 0))
        except: pass
        d = str(c.get('disease_problem', '')).strip()
        if d: dc[d] = dc.get(d, 0) + 1
    best = max(dc, key=dc.get) if dc else 'N/A'
    avg  = round(sum(ratings) / len(ratings), 2) if ratings else 0
    return pd.DataFrame({
        'KPI': ['Total cases', 'Successful (4-5)',
                'Avg success rating', 'Most common problem', 'Total points'],
        'Value': [len(cases), sum(1 for r in ratings if r >= 4), avg, best, points]
    })

# Adds a new before and after case
def add_gallery_case(plant_type, disease, before_img, after_img,
                     treatment, result_txt, success_rating):
    if not plant_type or not disease or before_img is None or after_img is None or not treatment:
        return ("Please fill all fields and upload both images.",
                _cases_to_df([]), _get_kpis([]))
    try:
        pts, badge = _calc_points(treatment, result_txt)
        case = {
            'date':             datetime.now().strftime('%Y-%m-%d %H:%M'),
            'plant_type':       plant_type.strip(),
            'disease_problem':  disease.strip(),
            'treatment':        treatment.strip(),
            'result':           (result_txt or '').strip(),
            'success_rating':   int(success_rating) if success_rating else 3,
            'points':           pts,
            'badge':            badge,
            'before_image':     _img_to_b64(before_img),
            'after_image':      _img_to_b64(after_img)
        }
        _save_case(case)
        updated = _load_cases()
        return (f"Case saved! +{pts} points - Badge: {badge}",
                _cases_to_df(updated), _get_kpis(updated))
    except Exception as e:
        return f"Error: {e}", _cases_to_df([]), _get_kpis([])

# Searches saved gallery cases
def search_gallery_cases(query):
    try:
        cases = _load_cases()
    except Exception:
        return "Could not load from Firebase.", pd.DataFrame(), []
    if not cases:
        return "No cases saved yet.", pd.DataFrame(), []
    matched = _rank_cases(query, cases)
    if not matched:
        return "No matching cases found.", pd.DataFrame(), []
    return f"Found {len(matched)} case(s):", _cases_to_df(matched), _cases_gallery(matched)

# Reloads the gallery data from Firebase
def refresh_gallery():
    try:   cases = _load_cases()
    except Exception: cases = []
    return _cases_to_df(cases), _get_kpis(cases)

print("Gallery functions ready (Firebase storage, no HF comparison)")


Gallery functions ready (Firebase storage, no HF comparison)


## Chatbot Backend

In this part we handle the chatbot conversation.
The chatbot sends the user's message together with the recent conversation to Gemini and returns the generated reply.

In [20]:
# Sends the user message to Gemini and updates the chat history
def chat_with_blueberry_bot(user_message, history):
    if not user_message.strip():
        return "", history
    ctx = ""
    for msg in (history or [])[-10:]:
        role    = msg.get('role', '')
        content = msg.get('content', '')
        ctx += f"{role.capitalize()}: {content}\n"
    prompt = (
        "You are a helpful AI assistant for 'Blueberry Smart Care'.\n"
        "Help with: plant health, diseases, sensor readings, growing tips.\n"
        f"Previous conversation:\n{ctx}\n"
        f"User: {user_message}\n"
        "Answer in simple, practical English. Be concise."
    )
    try:
        answer = client.models.generate_content(
            model="gemini-2.5-flash", contents=prompt).text
    except Exception as e:
        answer = f"Sorry, could not process your question. Error: {e}"
    history = list(history or [])
    history.append({"role": "user",      "content": user_message})
    history.append({"role": "assistant", "content": answer})
    return "", history

print("Chatbot ready")


Chatbot ready


## CSS Styles

In [21]:
custom_css = '''
/* Base */
.gradio-container, main {
    background: #F6F4FB !important; color: #1F1B2E !important;
    font-family: "Inter","Segoe UI",sans-serif !important; padding: 0 !important;
}
[role="tablist"] {
    height: 0 !important; min-height: 0 !important; overflow: hidden !important;
    opacity: 0 !important; margin: 0 !important; padding: 0 !important;
}
/* Home */
.home-hero { width:100%;height:100vh;min-height:650px;overflow:hidden;position:relative;margin:0; }
.home-bg   { width:100%;height:100%;object-fit:cover;object-position:center;display:block;
             filter:saturate(1.1) contrast(1.05); }
.home-overlay {
    position:absolute;inset:0;
    background:radial-gradient(circle at 30% 30%,rgba(91,60,196,.20),transparent 35%),
               linear-gradient(rgba(8,10,30,.45),rgba(8,10,30,.70));
    display:flex;flex-direction:column;justify-content:center;align-items:center;
    padding:clamp(24px,5vw,50px);text-align:center;
}
.home-logo { font-size:clamp(48px,6vw,76px);margin-bottom:8px; }
.home-overlay h1 { color:white !important;font-size:clamp(38px,6vw,72px);
    font-weight:900;margin:0;text-shadow:0 6px 18px rgba(0,0,0,.45); }
.home-overlay p  { color:white !important;font-size:clamp(16px,2vw,24px);
    max-width:760px;line-height:1.55;margin-top:22px;text-shadow:0 3px 12px rgba(0,0,0,.55); }
.home-buttons { display:flex;gap:24px;margin-top:40px;flex-wrap:wrap;justify-content:center; }
.home-action-btn {
    min-width:min(240px,90vw);padding:16px 26px;border-radius:22px;
    font-size:clamp(14px,2vw,18px);font-weight:850;color:white !important;
    cursor:pointer;backdrop-filter:blur(14px);transition:all .25s ease;
}
.purple-btn { background:rgba(91,60,196,.30)  !important;border:2px solid rgba(167,139,250,.85) !important; }
.blue-btn   { background:rgba(37,99,235,.28)   !important;border:2px solid rgba(96,165,250,.9)   !important; }
.green-btn  { background:rgba(5,150,105,.30)   !important;border:2px solid rgba(52,211,153,.85)  !important; }
.orange-btn { background:rgba(217,119,6,.30)   !important;border:2px solid rgba(251,191,36,.85)  !important; }
.teal-btn   { background:rgba(13,148,136,.30)  !important;border:2px solid rgba(45,212,191,.85)  !important; }
.home-action-btn:hover { transform:translateY(-4px);background:rgba(255,255,255,.20) !important; }
.home-footer {
    position:absolute;bottom:0;left:0;right:0;display:flex;gap:18px;
    justify-content:center;align-items:center;color:white !important;
    font-size:clamp(12px,1.4vw,16px);padding:18px;
    background:rgba(8,10,30,.26);backdrop-filter:blur(10px);
    border-top:1px solid rgba(255,255,255,.15);flex-wrap:wrap;
}
.home-footer div,.home-footer span { color:white !important; }
/* Screens */
.screen-wrapper { padding:28px; }
#title-box {
    background:linear-gradient(135deg,#FFFFFF,#EFE9FF);padding:34px 30px;
    border-radius:28px;margin-bottom:24px;
    box-shadow:0 14px 35px rgba(76,42,133,.12);border:1px solid #E6DCF8;
}
#title-box h1 { font-size:42px;margin:0;font-weight:850;color:#2F195F; }
#title-box p  { font-size:18px;margin-top:10px;color:#6B5A86; }
.helper-box {
    background:#FFFFFF;color:#4C3A67;border-radius:20px;
    padding:16px 20px;margin:12px 0 20px;font-size:15px;font-weight:500;
    border:1px solid #E6DCF8;box-shadow:0 8px 20px rgba(47,25,95,.06);
}
.block,.form,.panel {
    background:#FFFFFF !important;border-radius:22px !important;
    border:1px solid #E8E2F2 !important;
    box-shadow:0 10px 28px rgba(47,25,95,.08) !important;
    color:#1F1B2E !important;overflow:hidden !important;
}
label,span,p { color:#1F1B2E !important; }
/* Buttons */
button { background:#FFFFFF !important;color:#3B2474 !important;
    border:1px solid #DDD3F3 !important;border-radius:18px !important;font-weight:700 !important; }
button.primary, #load-button {
    background:linear-gradient(135deg,#5B3CC4,#7C5CE6) !important;
    color:white !important;border:none !important;border-radius:18px !important;
    font-weight:800 !important;box-shadow:0 10px 22px rgba(91,60,196,.25) !important;
}
.back-home-btn {
    background:#FFFFFF !important;color:#2F195F !important;border:1px solid #DDD3F3 !important;
    border-radius:16px !important;padding:12px 18px !important;
    font-weight:800 !important;margin-bottom:18px !important;cursor:pointer !important;
}
input,textarea,select {
    background:#FFFFFF !important;color:#1F1B2E !important;border-color:#DDD3F3 !important;
}
input[type="range"] { accent-color:#5B3CC4 !important; }
/* RADIO SELECTION - pill buttons with visible filled circle */
.sensor-radio fieldset { border:none !important; gap:12px !important; }
.sensor-radio fieldset label {
    background:#FFFFFF !important;
    border:3px solid #7C5CE6 !important;
    border-radius:50px !important;
    padding:10px 22px !important;
    min-height:unset !important;
    cursor:pointer !important;
    transition:all .2s ease !important;
    display:inline-flex !important;
    align-items:center !important;
    gap:8px !important;
    font-weight:700 !important;
    color:#2F195F !important;
    box-shadow:0 3px 10px rgba(91,60,196,.18) !important;
}
.sensor-radio fieldset label:has(input[type="radio"]:checked) {
    background:linear-gradient(135deg,#5B3CC4,#7C5CE6) !important;
    border-color:#5B3CC4 !important;
    color:#FFFFFF !important;
    box-shadow:0 5px 16px rgba(91,60,196,.40) !important;
    transform:translateY(-1px) !important;
}
.sensor-radio fieldset label:has(input[type="radio"]:checked) span {
    color:#FFFFFF !important;
}
/* Custom circle — visible before the label text */
.sensor-radio fieldset input[type="radio"] {
    appearance:none !important;
    -webkit-appearance:none !important;
    width:14px !important;
    height:14px !important;
    border:2px solid #7C5CE6 !important;
    border-radius:50% !important;
    background:#FFFFFF !important;
    display:inline-block !important;
    flex-shrink:0 !important;
    transition:all .2s ease !important;
    position:relative !important;
}
/* Filled circle when selected */
.sensor-radio fieldset label:has(input[type="radio"]:checked) input[type="radio"] {
    background:#FFFFFF !important;
    border-color:#FFFFFF !important;
    box-shadow:inset 0 0 0 3px #5B3CC4 !important;
}
/* Hide only Gradio's own SVG/div indicator (keep our custom input visible) */
.sensor-radio fieldset label svg { display:none !important; }
.sensor-radio fieldset label > div:first-child { display:none !important; }
/* Status cards */
.status-card {
    background:#FFFFFF;border-radius:26px;padding:26px;min-height:280px;
    box-shadow:0 14px 32px rgba(47,25,95,.10);border:1px solid #E8E2F2;
}
.status-icon     { font-size:38px;margin-bottom:10px; }
.status-title    { font-size:22px;font-weight:850;margin-bottom:8px; }
.status-value    { font-size:48px;font-weight:900;color:#2F195F;margin-bottom:8px; }
.status-subtitle { color:#7A6B91;font-size:14px;margin-bottom:18px; }
.status-row      { display:flex;justify-content:space-between;padding:11px 0;
                   border-top:1px solid #EFEAF7; }
.recommendation-box { margin-top:18px;padding:15px;border-radius:18px;
    background:#FFFFFF;color:#2F195F;font-weight:700; }
.healthy-card  { background:#F1FBF4; }
.warning-card  { background:#FFF8E6; }
.critical-card { background:#FFF0F0; }
/* Dashboard */
.dashboard-grid {
    display:grid;grid-template-columns:repeat(4,1fr);gap:18px;margin-bottom:20px;
}
.dashboard-card {
    background:#FFFFFF;border-radius:24px;padding:22px;text-align:center;
    box-shadow:0 10px 24px rgba(47,25,95,.08);border:1px solid #E8E2F2;
}
.dashboard-icon     { font-size:32px;margin-bottom:10px; }
.dashboard-title    { font-size:16px;font-weight:700;color:#6B5A86; }
.dashboard-value    { font-size:34px;font-weight:900;color:#2F195F;margin-top:8px; }
.dashboard-subtitle { font-size:13px;color:#8B7CA5;margin-top:8px; }
.health-score-card  { background:linear-gradient(135deg,#5B3CC4,#7C5CE6); }
.health-score-card .dashboard-title,
.health-score-card .dashboard-value,
.health-score-card .dashboard-subtitle { color:white !important; }
.dashboard-recommendation {
    background:#FFFFFF;border-radius:24px;padding:22px;margin-top:18px;border:1px solid #E8E2F2;
}
.gamification-box { margin-top:18px;padding:16px;border-radius:18px;
    background:#F3EEFF;border:1px solid #E6DCF8; }
th { background:#F3EEFF !important;color:#2F195F !important;font-weight:800 !important; }
/* Search */
.search-result-card { background:#FFFFFF;border:1px solid #E8E2F2;border-radius:20px;
    padding:22px;margin-bottom:16px;box-shadow:0 8px 20px rgba(47,25,95,.07); }
.search-result-header { display:flex;justify-content:space-between;
    align-items:center;margin-bottom:10px; }
.doc-id         { background:#F3EEFF;color:#5B3CC4;padding:4px 12px;
    border-radius:20px;font-weight:800;font-size:13px; }
.relevance-score { color:#059669;font-weight:800;font-size:14px; }
.search-result-title { font-size:18px;font-weight:850;color:#2F195F;margin-bottom:6px; }
.search-result-meta  { font-size:13px;color:#7A6B91;margin-bottom:12px; }
.relevance-bar  { font-family:monospace;font-size:13px;color:#5B3CC4;
    letter-spacing:1px;margin-bottom:10px; }
.search-excerpt { background:#FAFAFA;border-left:3px solid #5B3CC4;padding:10px 14px;
    border-radius:0 10px 10px 0;font-size:14px;color:#444;margin-bottom:12px; }
.doi-link   { color:#5B3CC4;font-weight:700;text-decoration:none;font-size:14px; }
.search-header { margin-bottom:20px;padding:20px;background:#F3EEFF;border-radius:16px; }
.search-header h2 { margin:0 0 8px;color:#2F195F !important;font-size:22px; }
.search-header p  { margin:0;color:#5B3CC4 !important; }
@media(max-width:900px) { .dashboard-grid { grid-template-columns:repeat(2,1fr); } }
@media(max-width:700px) {
    .home-buttons { flex-direction:column;width:100%;align-items:center; }
    .dashboard-grid { grid-template-columns:1fr; }
}
'''
print("CSS ready")


CSS ready


## Main Gradio Application

In this part we build the full Gradio interface.
The app includes the home page, image upload, sensors, search, dashboard, gallery, and chatbot screens.

In [22]:
# Build all HTML blocks separately to avoid nested quote issues
_HOME = (
    "<div class='home-hero'>"
    "<img class='home-bg' src='https://i.pinimg.com/736x/7f/73/21/7f73213f3f1869402396fa8c8d4aab10.jpg'>"
    "<div class='home-overlay'>"
    ""
    "<h1>Blueberry Smart Care</h1>"
    "<p>Monitor plant health, analyse sensor trends, and receive AI-powered recommendations.</p>"
    "<div class='home-buttons'>"
    "<button class='home-action-btn green-btn' onclick=\"document.querySelectorAll('button[role=tab]')[1].click()\">\U0001f4f7 Plant Images</button>"
    "<button class='home-action-btn purple-btn' onclick=\"document.querySelectorAll('button[role=tab]')[2].click()\">\U0001f4e1 Sensors</button>"
    "<button class='home-action-btn orange-btn' onclick=\"document.querySelectorAll('button[role=tab]')[3].click()\">\U0001f50d Search Papers</button>"
    "<button class='home-action-btn blue-btn' onclick=\"document.querySelectorAll('button[role=tab]')[4].click()\">\U0001f4ca Dashboard</button>"
    "<button class='home-action-btn teal-btn' onclick=\"document.querySelectorAll('button[role=tab]')[5].click()\">\U0001f33f Gallery</button>"
    "<button class='home-action-btn purple-btn' onclick=\"document.querySelectorAll('button[role=tab]')[6].click()\">\U0001f916 Chatbot</button>"
    "</div>"
    "<div class='home-footer'>"
    "<div>\U0001f4e1 Real-Time IoT Monitoring</div><span>·</span>"
    "<div>\U0001f9e0 RAG + AI Diagnosis</div><span>·</span>"
    "<div>\U0001f33f Better Plant Health</div>"
    "</div></div></div>"
)

_IMG_HDR = (
    "<div class='screen-wrapper'>"
    "<button class='back-home-btn' onclick=\"document.querySelectorAll('button[role=tab]')[0].click()\">← Back Home</button>"
    "<div id='title-box'><h1>\U0001f4f7 Plant Image Upload</h1></div>"
    "<div class='helper-box'>\U0001f4a1 Upload a photo and add optional notes. "
    "The AI model identifies the plant condition and Gemini provides personalised advice.</div>"
    "</div>"
)

_SEN_HDR = (
    "<div class='screen-wrapper'>"
    "<button class='back-home-btn' onclick=\"document.querySelectorAll('button[role=tab]')[0].click()\">← Back Home</button>"
    "<div id='title-box'><h1>\U0001fad0 Blueberry Plant Monitor</h1>"
    "<p>Real-Time Cloud Monitoring & Health Analysis</p></div>"
    "<div class='helper-box'>\U0001f4a1 Select a sensor type to view live readings. "
    "If the live server is unavailable, last saved readings are displayed automatically.</div></div>"
)

_SCH_HDR = (
    "<div class='screen-wrapper'>"
    "<button class='back-home-btn' onclick=\"document.querySelectorAll('button[role=tab]')[0].click()\">← Back Home</button>"
    "<div id='title-box'><h1>\U0001f50d Research Paper Search</h1>"
    "<p>RAG-powered search across 5 academic blueberry papers</p></div>"
    "<div class='helper-box'>\U0001f4a1 Enter a query about blueberry diseases, cultivation, or breeding. "
    "The RAG system retrieves the most relevant paper chunks and Gemini synthesises a smart answer.</div></div>"
)

_DSH_HDR = (
    "<div class='screen-wrapper'>"
    "<button class='back-home-btn' onclick=\"document.querySelectorAll('button[role=tab]')[0].click()\">← Back Home</button>"
    "<div id='title-box'><h1>\U0001fad0 My Blueberry Dashboard</h1>"
    "<p>Overall plant condition, AI agent analysis, sensor summary and gamification progress</p></div></div>"
)

_GAL_HDR = (
    "<div class='screen-wrapper'>"
    "<button class='back-home-btn' onclick=\"document.querySelectorAll('button[role=tab]')[0].click()\">← Back Home</button>"
    "<div id='title-box'><h1>\U0001f33f Before & After Gallery</h1>"
    "<p>Share plant treatment cases and learn from the community.</p></div>"
    "<div class='helper-box'>\U0001f4a1 Fill in the treatment details below and upload before/after photos. "
    "All shared cases are visible to everyone and searchable.</div></div>"
)

_BOT_HDR = (
    "<div class='screen-wrapper'>"
    "<button class='back-home-btn' onclick=\"document.querySelectorAll('button[role=tab]')[0].click()\">← Back Home</button>"
    "<div id='title-box'><h1>\U0001f916 Blueberry AI Assistant</h1>"
    "<p>Ask anything about your blueberry plants</p></div>"
    "<div class='helper-box'>\U0001f4a1 Ask about diseases, sensor readings, growing tips, or plant health.</div></div>"
)

with gr.Blocks() as app:
    with gr.Tabs():

        # HOME
        with gr.Tab('Home', id='home'):
            gr.HTML(_HOME)

        # SCREEN 1: Images
        with gr.Tab('Images', id='images') as images_tab:
            gr.HTML(_IMG_HDR)
            # Image upload section
            with gr.Row(equal_height=True):
                with gr.Column(scale=1):
                    img_input = gr.Image(label='Plant Photo', type='filepath')
                with gr.Column(scale=1):
                    plant_name_in = gr.Textbox(label='Plant Name / ID',
                                               placeholder='e.g. Blueberry Bush #1')
                    notes_in      = gr.Textbox(label='Observations / Notes', lines=5,
                                               placeholder='Describe what you see...')
            # Analyse button
            upload_btn = gr.Button('🔍 Analyse & Save', variant='primary')
            upload_msg = gr.Textbox(label='Status', interactive=False, visible=False)
            # AI analysis and gallery
            with gr.Row():
                with gr.Column(scale=1):
                    ai_output = gr.HTML(label='AI Analysis')
                with gr.Column(scale=1):
                    img_gallery = gr.Gallery(label='Session Images', columns=2, height=340)
            # Log section
            gr.Markdown("### Image Upload History")
            refresh_log_btn = gr.Button('🔄 Refresh Log', variant='primary')
            images_table    = gr.Dataframe(label='Image Log', value=load_image_log(),
                                           interactive=False)
            upload_btn.click(
                fn=upload_plant_image,
                inputs=[img_input, plant_name_in, notes_in],
                outputs=[upload_msg, img_gallery, ai_output, images_table]
            )
            refresh_log_btn.click(fn=load_image_log, inputs=[], outputs=[images_table], show_progress=True)

        # SCREEN 2: Sensors
        with gr.Tab('Sensors', id='sensors'):
            gr.HTML(_SEN_HDR)
            with gr.Row():
                feed_input = gr.Radio(
                    choices=[
                        ('Humidity', 'humidity'),
                        ('Soil Moisture', 'soil'),
                        ('Temperature', 'temperature')
                    ],
                    value='humidity',
                    label='Choose Sensor',
                    elem_classes=['sensor-radio']
                )
                limit_input = gr.Slider(minimum=1, maximum=100, value=30, step=1,
                                        label='Number of Samples')
            with gr.Row():
                check_button = gr.Button('Load Sensor Data', variant='primary',
                                         elem_id='load-button')
                reset_button = gr.Button('Reset')
            with gr.Row():
                with gr.Column(scale=1):
                    status_output = gr.HTML(label='Sensor Status')
                with gr.Column(scale=2):
                    plot_output = gr.Plot(label='Trend Graph')
            table_output = gr.Dataframe(label='Recent Measurements')
            check_button.click(fn=get_sensor_history,
                               inputs=[feed_input, limit_input],
                               outputs=[status_output, plot_output, table_output],
                               show_progress=True)
            reset_button.click(fn=lambda: ('', None, None), inputs=[],
                               outputs=[status_output, plot_output, table_output])

        # SCREEN 3: Search / RAG
        with gr.Tab('Search', id='search'):
            gr.HTML(_SCH_HDR)
            with gr.Row():
                search_query = gr.Textbox(
                    label='Search Query',
                    placeholder='e.g. botrytis disease treatment blueberry',
                    scale=4
                )
                n_results_in = gr.Slider(minimum=1, maximum=5, value=3, step=1,
                                         label='Results', scale=1)
            with gr.Row():
                search_btn = gr.Button('Search Papers', variant='primary')
                clear_btn  = gr.Button('Clear')
            search_output = gr.HTML(label='Search Results')
            search_btn.click(fn=run_search, inputs=[search_query, n_results_in],
                             outputs=[search_output])
            clear_btn.click(fn=lambda: ('', ''), inputs=[],
                            outputs=[search_query, search_output])

        # SCREEN 4: Dashboard
        with gr.Tab('Dashboard', id='dashboard') as dash_tab:
            gr.HTML(_DSH_HDR)
            dashboard_range = gr.Radio(
                choices=[('Today', 'today'), ('Yesterday', 'yesterday'), ('Last 7 Days', 'week')],
                value='today', label='Dashboard View',
                elem_classes=['sensor-radio']
            )
            load_dash_btn = gr.Button('Load Dashboard', variant='primary')
            cards_output  = gr.HTML()
            dash_plot     = gr.Plot(label='Sensor Trends Comparison')
            rec_output    = gr.HTML()
            load_dash_btn.click(fn=load_dashboard, inputs=[dashboard_range],
                                outputs=[cards_output, dash_plot, rec_output],
                                show_progress=True)

        # SCREEN 5: Gallery
        with gr.Tab('Gallery', id='gallery') as gallery_tab:
            gr.HTML(_GAL_HDR)
            # Add case
            gr.HTML("<div style='font-size:17px;font-weight:800;color:#2F195F;margin:8px 0 12px;'>\U0001f4dd Share a Treatment Case</div>")
            # Plant details
            with gr.Row():
                g_plant   = gr.Textbox(label='Plant Type', placeholder='e.g. Blueberry')
                g_disease = gr.Textbox(label='Disease / Problem', placeholder='e.g. Yellow leaves')
            with gr.Row():
                g_treat  = gr.Textbox(label='Treatment Used', lines=3)
                g_result = gr.Textbox(label='Result / Recommendation', lines=3)
            g_rating = gr.Number(label='Success Rating (1-5)', value=3,
                                 minimum=1, maximum=5, step=1, precision=0)
            # Before/After images: side by side, equal size
            with gr.Row(equal_height=True):
                with gr.Column(scale=1):
                    g_before = gr.Image(label='📷 Before Treatment', type='filepath')
                with gr.Column(scale=1):
                    g_after  = gr.Image(label='📷 After Treatment', type='filepath')
            # Share button: full width directly below the images
            add_btn = gr.Button('\U0001f4e4 Share Case', variant='primary')
            add_msg = gr.Textbox(label='Status', interactive=False)
            gr.HTML("<hr style='border:none;border-top:2px solid #E6DCF8;margin:22px 0;'/>")
            # Search
            gr.HTML("<div style='font-size:17px;font-weight:800;color:#2F195F;margin-bottom:12px;'>\U0001f50d Search Similar Cases</div>")
            g_search_q = gr.Textbox(
                label='Search by plant / disease / treatment',
                placeholder='e.g. blueberry, yellow leaves',
            )
            g_srch_btn = gr.Button('\U0001f50d Search', variant='primary')
            g_srch_msg = gr.Textbox(label='Status', interactive=False, visible=False)
            with gr.Row():
                with gr.Column(scale=1):
                    g_srch_tbl = gr.Dataframe(label='Similar Cases', interactive=False)
                with gr.Column(scale=1):
                    g_srch_gal = gr.Gallery(label='Before & After Images', columns=2, height='auto')
            gr.HTML("<hr style='border:none;border-top:2px solid #E6DCF8;margin:22px 0;'/>")
            # All cases (auto-loaded)
            gr.HTML("<div style='font-size:17px;font-weight:800;color:#2F195F;margin-bottom:12px;'>\U0001f4da All Treatment Cases</div>")
            refresh_gal_btn = gr.Button('\U0001f504 Refresh Cases', variant='primary')
            all_cases_tbl   = gr.Dataframe(label='All Cases', interactive=False,
                                           value=refresh_gallery()[0])
            kpi_tbl         = gr.Dataframe(label='Gallery KPIs', interactive=False,
                                           value=refresh_gallery()[1])

            add_btn.click(
                fn=add_gallery_case,
                inputs=[g_plant, g_disease, g_before, g_after,
                        g_treat, g_result, g_rating],
                outputs=[add_msg, all_cases_tbl, kpi_tbl]
            )
            g_srch_btn.click(
                fn=search_gallery_cases, inputs=[g_search_q],
                outputs=[g_srch_msg, g_srch_tbl, g_srch_gal]
            )
            refresh_gal_btn.click(
                fn=refresh_gallery, inputs=[],
                outputs=[all_cases_tbl, kpi_tbl]
            )

        # SCREEN 6: Chatbot
        with gr.Tab('Chatbot', id='chatbot'):
            gr.HTML(_BOT_HDR)
            # Input area: textbox then buttons below
            chat_input = gr.Textbox(
                placeholder='e.g. My humidity is 34%, is that OK?',
                label='Ask the Blueberry Assistant',
                lines=2
            )
            with gr.Row():
                chat_submit = gr.Button('💬 Send', variant='primary')
                chat_clear  = gr.Button('🗑 Clear')
            # Conversation below
            chatbot = gr.Chatbot(
                label='Conversation', height=480,
                show_label=False
            )
            chat_submit.click(
                fn=chat_with_blueberry_bot,
                inputs=[chat_input, chatbot],
                outputs=[chat_input, chatbot]
            )
            chat_input.submit(
                fn=chat_with_blueberry_bot,
                inputs=[chat_input, chatbot],
                outputs=[chat_input, chatbot]
            )
            chat_clear.click(fn=lambda: ([], ''), outputs=[chatbot, chat_input])

        # Auto-load: show data immediately on first open
        app.load(fn=load_image_log, inputs=[], outputs=[images_table])
        app.load(fn=lambda: load_dashboard('today'),
                 inputs=[], outputs=[cards_output, dash_plot, rec_output])
        # Re-load dashboard whenever user navigates to Dashboard tab
        dash_tab.select(fn=lambda: load_dashboard('today'),
                        inputs=[], outputs=[cards_output, dash_plot, rec_output])

app.launch(theme=gr.themes.Soft(), css=custom_css)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://109c419dd9266d7ee3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
